In [2]:
import os
from urllib.parse import urlparse
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans, BisectingKMeans
from pyspark.ml.evaluation import ClusteringEvaluator
import time
import pathlib
import csv
import numpy as np
import pandas as pd
from datetime import datetime
import sys
from io import StringIO

# ========================================
# CONFIGURATION
# ========================================

TRACK = "A"
PATH_CHOICE = "CLUSTERING"
STUDENT_NAMES = ["Bibawandaogo"]

print("="*70)
print("DE2 LAB 3 PRACTICE - INSTRUMENTED ITERATIVE WORKLOAD")
print("="*70)
print(f"Track: {TRACK}")
print(f"Path: {PATH_CHOICE}")
print(f"Students: {STUDENT_NAMES}")
print("="*70 + "\n")

# ========================================
# SPARK INIT
# ========================================

DE2_SPARK_DRIVER_HOST = os.environ.get("DE2_SPARK_DRIVER_HOST", "127.0.0.1")
DE2_SPARK_BIND_ADDRESS = os.environ.get("DE2_SPARK_BIND_ADDRESS", "0.0.0.0")
os.environ.setdefault("SPARK_LOCAL_IP", DE2_SPARK_DRIVER_HOST)

def show_spark_ui(spark_session):
    ui_url = spark_session.sparkContext.uiWebUrl
    print("Spark version:", spark_session.version)
    if ui_url:
        ui_port = urlparse(ui_url).port or 4040
        print("Spark UI:", ui_url)
        print("Spark UI (WSL/Windows):", f"http://localhost:{ui_port}")
    else:
        print("Spark UI: not available")

spark = SparkSession.builder \
    .appName("DE2-Lab3-Iterative-Clustering") \
    .master("local[*]") \
    .config("spark.driver.host", DE2_SPARK_DRIVER_HOST) \
    .config("spark.driver.bindAddress", DE2_SPARK_BIND_ADDRESS) \
    .config("spark.ui.bindAddress", DE2_SPARK_BIND_ADDRESS) \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

show_spark_ui(spark)

# ========================================
# CREATE DIRECTORIES
# ========================================

pathlib.Path("outputs/lab3").mkdir(parents=True, exist_ok=True)
pathlib.Path("proof").mkdir(parents=True, exist_ok=True)

# ========================================
# PART 1: BUILD SYNTHETIC DATASET (Track A - Esports)
# ========================================

print("\n=== PART 1: DATA PREPARATION ===\n")

np.random.seed(42)
n_samples = 5000
n_features = 5

print("Generating synthetic Esports hero stats (Track A)...")

cluster1 = np.random.normal(loc=[60, 35, 2.5, 1.2, 0.8], scale=3, size=(n_samples//3, n_features))
cluster2 = np.random.normal(loc=[75, 50, 3.5, 2.0, 1.2], scale=3, size=(n_samples//3, n_features))
cluster3 = np.random.normal(loc=[45, 20, 1.5, 0.5, 0.3], scale=3, size=(n_samples - 2*(n_samples//3), n_features))

data = np.vstack([cluster1, cluster2, cluster3])
np.random.shuffle(data)

schema = StructType([
    StructField("id", LongType(), False),
    StructField("win_rate", DoubleType(), False),
    StructField("pick_rate", DoubleType(), False),
    StructField("kda", DoubleType(), False),
    StructField("damage_dealt", DoubleType(), False),
    StructField("kill_participation", DoubleType(), False),
])

rows = [
    (int(i), float(data[i, 0]), float(data[i, 1]), float(data[i, 2]),
     float(data[i, 3]), float(data[i, 4]))
    for i in range(len(data))
]

df_raw = spark.createDataFrame(rows, schema=schema)

print(f"Total samples: {df_raw.count()}")
print("Sample rows:")
df_raw.show(5)

# ========================================
# PART B.1: FEATURE PREPARATION
# ========================================

print("\n=== PART B.1: FEATURE PREPARATION ===\n")

feature_cols = ["win_rate", "pick_rate", "kda", "damage_dealt", "kill_participation"]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withMean=True,
    withStd=True
)

df_assembled = assembler.transform(df_raw)
df_features = scaler.fit(df_assembled).transform(df_assembled)

print(f"Feature rows: {df_features.count()}")
print("Normalized features:")
df_features.select("features").show(3, truncate=50)

df_features.cache()
df_features.count()



# ========================================
# PART B.2: KMEANS / BISECTINGKMEANS SWEEPS
# ========================================

print("\n=== PART B.2: CLUSTERING SWEEPS ===\n")

evaluator = ClusteringEvaluator(
    predictionCol="prediction",
    featuresCol="features",
    metricName="silhouette"
)

k_values = [3, 5, 8]
sweep_results = []
timestamp_start = datetime.now().isoformat()

print("Running KMeans and BisectingKMeans sweeps...\n")

for algo_cls, algo_name in [(KMeans, "KMeans"), (BisectingKMeans, "BisectingKMeans")]:
    for k in k_values:
        print(f"Running {algo_name} with k={k}")

        t0 = time.time()

        try:
            if algo_name == "KMeans":
                model = algo_cls(k=k, featuresCol="features", seed=42, maxIter=10).fit(df_features)
                preds = model.transform(df_features)
                sil = evaluator.evaluate(preds)
            else:
                model = algo_cls(k=k, featuresCol="features", seed=42).fit(df_features)
                preds = model.transform(df_features)
                sil = evaluator.evaluate(preds)

            elapsed = (time.time() - t0) * 1000

            sweep_results.append({
                "algorithm": algo_name,
                "k": k,
                "silhouette": sil,
                "elapsed_ms": elapsed
            })

            print(f"  Silhouette: {sil:.4f}, Time: {elapsed:.2f}ms\n")

        except Exception as e:
            print(f"  Error with {algo_name} k={k}: {str(e)[:100]}")
            print(f"  Skipping this configuration\n")
            continue

sweep_df = pd.DataFrame(sweep_results)
print("Sweep Results:")
print(sweep_df.to_string(index=False))

if len(sweep_df) == 0:
    print("ERROR: No successful clustering runs!")
    spark.stop()
    exit(1)

# Find best k
best_row = sweep_df.loc[sweep_df['silhouette'].idxmax()]
best_k = int(best_row['k'])
best_algo = best_row['algorithm']

print(f"\nBest configuration: {best_algo} with k={best_k} (silhouette={best_row['silhouette']:.4f})")

# ========================================
# PART B.3: SEED STABILITY ANALYSIS
# ========================================

print("\n=== PART B.3: SEED STABILITY ANALYSIS ===\n")

print(f"Testing seed stability for {best_algo} with k={best_k}...\n")

seed_results = []
n_seeds = 5

for seed_idx, seed in enumerate(range(42, 42 + n_seeds)):
    print(f"Seed {seed_idx + 1}/{n_seeds} (seed={seed})")

    t0 = time.time()

    try:
        if best_algo == "KMeans":
            model = KMeans(k=best_k, featuresCol="features", seed=seed, maxIter=10).fit(df_features)
        else:
            model = BisectingKMeans(k=best_k, featuresCol="features", seed=seed).fit(df_features)

        preds = model.transform(df_features)
        sil = evaluator.evaluate(preds)
        elapsed = (time.time() - t0) * 1000

        seed_results.append({
            "seed": seed,
            "k": best_k,
            "silhouette": sil,
            "elapsed_ms": elapsed
        })

        print(f"  Silhouette: {sil:.4f}, Time: {elapsed:.2f}ms\n")

    except Exception as e:
        print(f"  Error with seed {seed}: {str(e)[:100]}")
        print(f"  Skipping this seed\n")
        continue

seed_df = pd.DataFrame(seed_results)

if len(seed_df) > 0:
    print(f"Seed Stability Statistics (k={best_k}):")
    print(f"  Mean:   {seed_df['silhouette'].mean():.4f}")
    print(f"  Std:    {seed_df['silhouette'].std():.4f}")
    print(f"  Min:    {seed_df['silhouette'].min():.4f}")
    print(f"  Max:    {seed_df['silhouette'].max():.4f}")
    print(f"  CV:     {(seed_df['silhouette'].std() / seed_df['silhouette'].mean() * 100):.2f}%\n")
else:
    print("Warning: No successful seed stability runs!")

# ========================================
# PART B.4: PARTITIONING EXPERIMENT
# ========================================

print("\n=== PART B.4: PARTITIONING EXPERIMENT ===\n")

partitioning_results = []

print("--- Strategy A: DEFAULT PARTITIONING ---\n")

df_default = df_features

t0 = time.time()
try:
    if best_algo == "KMeans":
        model_default = KMeans(k=best_k, featuresCol="features", seed=42, maxIter=10).fit(df_default)
    else:
        model_default = BisectingKMeans(k=best_k, featuresCol="features", seed=42).fit(df_default)

    preds_default = model_default.transform(df_default)
    sil_default = evaluator.evaluate(preds_default)
    elapsed_default = (time.time() - t0) * 1000

    print(f"Silhouette: {sil_default:.4f}")
    print(f"Time: {elapsed_default:.2f}ms")
    print(f"Partitions: {df_default.rdd.getNumPartitions()}\n")

    partitioning_results.append({
        "strategy": "DEFAULT",
        "k": best_k,
        "algorithm": best_algo,
        "silhouette": sil_default,
        "elapsed_ms": elapsed_default,
        "n_partitions": df_default.rdd.getNumPartitions()
    })

    # Save model and predictions
    model_default.save("outputs/lab3/model_default")
    preds_default.select("id", "prediction").coalesce(1).write.mode("overwrite").parquet("outputs/lab3/predictions_default")

except Exception as e:
    print(f"Error in DEFAULT partitioning: {str(e)[:100]}\n")

print("--- Strategy B: OPTIMIZED PARTITIONING (32 partitions) ---\n")

n_partitions = 32
df_repartitioned = df_features.repartition(n_partitions, F.col("id"))

print(f"Repartitioned to {n_partitions} partitions")
print(f"Default partitions: {df_default.rdd.getNumPartitions()}")
print(f"Optimized partitions: {df_repartitioned.rdd.getNumPartitions()}\n")

t0 = time.time()
try:
    if best_algo == "KMeans":
        model_optimized = KMeans(k=best_k, featuresCol="features", seed=42, maxIter=10).fit(df_repartitioned)
    else:
        model_optimized = BisectingKMeans(k=best_k, featuresCol="features", seed=42).fit(df_repartitioned)

    preds_optimized = model_optimized.transform(df_repartitioned)
    sil_optimized = evaluator.evaluate(preds_optimized)
    elapsed_optimized = (time.time() - t0) * 1000

    print(f"Silhouette: {sil_optimized:.4f}")
    print(f"Time: {elapsed_optimized:.2f}ms")
    print(f"Partitions: {df_repartitioned.rdd.getNumPartitions()}\n")

    partitioning_results.append({
        "strategy": "OPTIMIZED",
        "k": best_k,
        "algorithm": best_algo,
        "silhouette": sil_optimized,
        "elapsed_ms": elapsed_optimized,
        "n_partitions": df_repartitioned.rdd.getNumPartitions()
    })

    # Save model and predictions
    model_optimized.save("outputs/lab3/model_optimized")
    preds_optimized.select("id", "prediction").coalesce(1).write.mode("overwrite").parquet("outputs/lab3/predictions_optimized")

except Exception as e:
    print(f"Error in OPTIMIZED partitioning: {str(e)[:100]}\n")

print("--- PARTITIONING COMPARISON ---\n")

partitioning_df = pd.DataFrame(partitioning_results)
print(partitioning_df.to_string(index=False))

if len(partitioning_df) == 2:
    speedup = partitioning_df.iloc[0]['elapsed_ms'] / partitioning_df.iloc[1]['elapsed_ms']
    improvement = (partitioning_df.iloc[0]['elapsed_ms'] - partitioning_df.iloc[1]['elapsed_ms']) / partitioning_df.iloc[0]['elapsed_ms'] * 100

    print(f"\nSpeedup: {speedup:.2f}x")
    print(f"Improvement: {improvement:.2f}%\n")
else:
    print("\nWarning: Could not compute speedup (not all partitioning runs succeeded)\n")


DE2 LAB 3 PRACTICE - INSTRUMENTED ITERATIVE WORKLOAD
Track: A
Path: CLUSTERING
Students: ['Bibawandaogo']

Spark version: 4.0.1
Spark UI: http://127.0.0.1:4041
Spark UI (WSL/Windows): http://localhost:4041

=== PART 1: DATA PREPARATION ===

Generating synthetic Esports hero stats (Track A)...
Total samples: 5000
Sample rows:
+---+------------------+------------------+-------------------+------------------+-------------------+
| id|          win_rate|         pick_rate|                kda|      damage_dealt| kill_participation|
+---+------------------+------------------+-------------------+------------------+-------------------+
|  0| 74.40021023064475|52.228709916300794|-2.7108206335401857|2.1235385508468045|  2.741025647379918|
|  1|  65.9170334372466| 36.39275371436901|  4.327406557160541| 2.917335918465305| 1.9911400096471998|
|  2| 64.24923537510088| 37.31971995999728|   2.02648747047922| 2.124876190916463|-1.0011570786133173|
|  3| 74.63607398583126| 52.78421601998207| 0.620449093

--- Strategy B: OPTIMIZED PARTITIONING (32 partitions) ---

Repartitioned to 32 partitions
Default partitions: 12
Optimized partitions: 32

Silhouette: 0.3385
Time: 2825.45ms
Partitions: 32

--- PARTITIONING COMPARISON ---

 strategy  k algorithm  silhouette  elapsed_ms  n_partitions
  DEFAULT  3    KMeans    0.329322 1402.841568            12
OPTIMIZED  3    KMeans    0.338525 2825.445890            32

Speedup: 0.50x
Improvement: -101.41%

